## OpenJobAI XGBoost Feedback Learning Evaluation

This notebook evaluates the machine learning component of **OpenJobAI**, an AI-powered job matching and resume tracking platform.

The purpose of this notebook is to show how the XGBoost feedback-learning model is connected to the full application workflow. The model uses feedback records stored in PostgreSQL and learns from resume-job matching features to predict whether a job is likely to be preferred by the user.

This notebook demonstrates the following workflow:

1. Load scored jobs from the FastAPI backend.
2. Create a bulk feedback dataset from existing scored jobs.
3. Save feedback records into PostgreSQL through the backend API.
4. Train the XGBoost model using feedback labels.
5. Evaluate the trained model using accuracy, precision, recall, F1 score, confusion matrix, and classification report.

This evaluation is part of the OpenJobAI MVP. The results are useful for verifying that the ML pipeline is working, but they should be treated as preliminary diagnostic results because the feedback dataset was generated for testing rather than collected from long-term real user behavior.


## 1. Loading Scored Jobs from the Backend

In this step, I load stored job records from the OpenJobAI FastAPI backend using the `/jobs` endpoint.

The backend returns jobs that were already fetched, processed, and stored in PostgreSQL. Each job record includes information such as job ID, title, company, location, ATS score, ML preference score, and recommended score.

For this test, I used **Resume ID 8**. Passing the resume ID allows the backend to return resume-specific scores for each job.

This step is important because the XGBoost model needs resume-job examples. Each job loaded here can later be connected with feedback labels such as **Interested** or **Not Interested**.

Expected output from this step:

- Backend status code should be `200`
- Jobs should be loaded successfully
- The job records should include score-related fields

In this run, the system loaded **723 jobs**, which became the base job pool for bulk feedback labeling.


In [1]:
import requests
import pandas as pd
import numpy as np

API_BASE_URL = "http://127.0.0.1:8000"
RESUME_ID = 8  # change this if your active resume ID is different

response = requests.get(
    f"{API_BASE_URL}/jobs",
    params={
        "limit": 1000,
        "offset": 0,
        "source_filter": "all",
        "freshness": "all",
        "relevant_only": "false",
        "resume_id": RESUME_ID,
    }
)

print("Status:", response.status_code)
print("URL:", response.url)
print(response.text[:500])

response.raise_for_status()
jobs = response.json()

jobs_df = pd.DataFrame([
    {
        "job_id": job.get("id"),
        "title": job.get("title"),
        "company": job.get("company"),
        "location": job.get("location"),
        "ats_score": job.get("ats_score"),
        "ml_preference_score": job.get("ml_preference_score"),
        "recommended_score": job.get("recommended_score"),
    }
    for job in jobs
])

print("Jobs loaded:", len(jobs_df))
jobs_df.head()

Status: 200
URL: http://127.0.0.1:8000/jobs?limit=1000&offset=0&source_filter=all&freshness=all&relevant_only=false&resume_id=8
[{"id":1618,"source":"indeed_apify","source_job_id":"dfbcc71690e00646","title":"Full-Stack Azure Data Engineer","company":"Telecommunications Development Corp.","location":"Laurel, MD 20709","description":"*Full-Stack Azure Data Engineer* Role Purpose: Develop, deploy, and maintain high-performance, cloud-native data pipelines. Key Responsibilities: * Build and scale ETL/ELT pipelines using Azure Databricks notebooks and Synapse scripts. * Implement standardized logging, error handling, and rest
Jobs loaded: 723


,job_id,title,company,location,ats_score,ml_preference_score,recommended_score
0,1618,Full-Stack Azure Data Engineer,Telecommunications Development Corp.,"Laurel, MD 20709",88,95.03,90
1,1051,"Big Data Developer- Pyspark, SQL, Scala",LTM,"Irving, TX",87,95.66,90
2,1714,Lead Software Engineer – Java Data Engineering,Strategic Staffing Solutions,"Plano, TX 75024",85,97.64,89
3,807,Data Analytics Engineer,Murdoch's Ranch & Home Supply,"Bozeman, MT 59715",86,94.65,89
4,1644,Azure Data Engineer,Bull City Talent Group,"Pennsylvania, United States",85,95.66,88


## 2. Creating Bulk Feedback Labels

The XGBoost model needs labeled training data before it can learn. In the real application, these labels would come from user actions such as clicking **Interested**, **Applied**, or **Not Interested** on the Matches page.

For this MVP test, I created a bulk feedback dataset from the existing scored jobs.

The labeling strategy was:

- Jobs with higher ATS-style scores were labeled as **positive examples**
- Jobs with lower ATS-style scores were labeled as **negative examples**

The code uses the ATS score as the main labeling signal because the ATS score exists before the ML model is trained. If the ATS score is missing, the code uses the recommended score as a fallback.

I used quantile-based selection:

- Top-scoring jobs were selected as positive examples
- Bottom-scoring jobs were selected as negative examples

For this run, I limited the dataset to:

| Label Type | Count |
|---|---:|
| Positive examples | 50 |
| Negative examples | 50 |

This creates a balanced feedback dataset for testing the XGBoost training pipeline.

Important note: this is **bulk pseudo-feedback** created for MVP testing. It proves that the ML pipeline is working, but it is not the same as long-term real user feedback.


In [2]:
jobs_for_training = jobs_df.copy()

# Prefer ATS score because it exists before ML training.
jobs_for_training["score_for_labeling"] = jobs_for_training["ats_score"]

# If ATS score is missing, use recommended score.
if jobs_for_training["score_for_labeling"].isna().all():
    jobs_for_training["score_for_labeling"] = jobs_for_training["recommended_score"]

jobs_for_training = jobs_for_training.dropna(
    subset=["job_id", "score_for_labeling"]
).copy()

jobs_for_training["job_id"] = jobs_for_training["job_id"].astype(int)

print("Jobs available for bulk labeling:", len(jobs_for_training))

# Use quantiles instead of fixed thresholds.
# Top 30% becomes positive, bottom 30% becomes negative.
positive_threshold = jobs_for_training["score_for_labeling"].quantile(0.70)
negative_threshold = jobs_for_training["score_for_labeling"].quantile(0.30)

positive_jobs = jobs_for_training[
    jobs_for_training["score_for_labeling"] >= positive_threshold
].copy()

negative_jobs = jobs_for_training[
    jobs_for_training["score_for_labeling"] <= negative_threshold
].copy()

# Limit bulk training size if needed.
MAX_POSITIVE = 50
MAX_NEGATIVE = 50

positive_jobs = positive_jobs.sort_values(
    "score_for_labeling", ascending=False
).head(MAX_POSITIVE)

negative_jobs = negative_jobs.sort_values(
    "score_for_labeling", ascending=True
).head(MAX_NEGATIVE)

print("Positive training examples:", len(positive_jobs))
print("Negative training examples:", len(negative_jobs))

display(positive_jobs[["job_id", "title", "company", "score_for_labeling"]].head(10))
display(negative_jobs[["job_id", "title", "company", "score_for_labeling"]].head(10))

Jobs available for bulk labeling: 723
Positive training examples: 50
Negative training examples: 50


,job_id,title,company,score_for_labeling
0,1618,Full-Stack Azure Data Engineer,Telecommunications Development Corp.,88
1,1051,"Big Data Developer- Pyspark, SQL, Scala",LTM,87
3,807,Data Analytics Engineer,Murdoch's Ranch & Home Supply,86
2,1714,Lead Software Engineer – Java Data Engineering,Strategic Staffing Solutions,85
4,1644,Azure Data Engineer,Bull City Talent Group,85
20,748,AI/ML Engineer,Cornerstone Defense,85
53,1701,Software Engineer III - Python Automation,JPMorganChase,85
80,804,Sr. Database Admin/Engineer - Exadata,Quevera LLC,85
79,838,Database Engineer III,BTS Software Solutions,85
84,674,Developer,Tata Consultancy Services,85


,job_id,title,company,score_for_labeling
722,1144,Full Stack Engineer (New Grad),Monarch Recruiters,58
208,1632,Senior Data Engineer,Volume Integration,60
715,1168,Full Stack Software Engineer 5 - Games Social ...,Netflix,60
721,740,"Software Engineer, Private Computing",OpenAI,60
717,1055,Unix Engineer with Python,Centraprise,60
718,882,Python Developer,Jobs via Dice,60
719,864,Backend Software Engineer - US Remote,Pragmatike,60
714,1192,"Software Engineer, Routing",Nuro,60
720,760,Accounting Analyst,HUB International,60
700,1139,Backend Engineer,Quindar,60


## 3. Saving Feedback Records to PostgreSQL

After selecting positive and negative job examples, I saved the feedback records through the backend `/feedback` API endpoint.

Each feedback record contains:

- Resume ID
- Job ID
- Feedback label

The positive examples were saved with the label:

`interested`

The negative examples were saved with the label:

`not_interested`

These labels are stored in PostgreSQL and later used by the XGBoost training endpoint.

This step connects the notebook evaluation to the actual application backend and database. Instead of training only inside the notebook, the feedback records are saved into the same storage system used by the web application.

In this run, the backend successfully created:

| Result | Count |
|---|---:|
| Created feedback records | 100 |
| Failed feedback records | 0 |

This confirms that the feedback-saving API and PostgreSQL storage workflow are working correctly.


In [3]:
created = []
failed = []

# Create positive feedback
for _, row in positive_jobs.iterrows():
    payload = {
        "resume_id": RESUME_ID,
        "job_id": int(row["job_id"]),
        "feedback_label": "interested",
    }

    response = requests.post(f"{API_BASE_URL}/feedback", json=payload)

    if response.status_code in [200, 201]:
        created.append(response.json())
    else:
        failed.append({
            "job_id": int(row["job_id"]),
            "label": "interested",
            "status": response.status_code,
            "response": response.text[:300],
        })

# Create negative feedback
for _, row in negative_jobs.iterrows():
    payload = {
        "resume_id": RESUME_ID,
        "job_id": int(row["job_id"]),
        "feedback_label": "not_interested",
    }

    response = requests.post(f"{API_BASE_URL}/feedback", json=payload)

    if response.status_code in [200, 201]:
        created.append(response.json())
    else:
        failed.append({
            "job_id": int(row["job_id"]),
            "label": "not_interested",
            "status": response.status_code,
            "response": response.text[:300],
        })

print("Created feedback records:", len(created))
print("Failed feedback records:", len(failed))

if failed:
    pd.DataFrame(failed).head()

Created feedback records: 100
Failed feedback records: 0


## 4. Verifying Stored Feedback Records

In this step, I reload feedback records from the backend to confirm that the feedback was saved correctly.

The feedback records are retrieved from the `/feedback` endpoint and converted into a DataFrame for inspection.

This validation step is important because the XGBoost model can only train if feedback records exist in the database. Earlier, when there were no feedback records, the model correctly returned that training could not continue. After adding feedback, the model now has supervised labels.

In this run, the feedback table contained **125 total feedback records**.

The feedback label distribution was:

| Feedback Label | Count |
|---|---:|
| not_interested | 66 |
| interested | 59 |

This distribution gives the model both positive and negative classes, which is required for binary supervised learning.

The feedback records connect a resume and a job together. Therefore, each feedback row becomes one labeled resume-job training example.


In [4]:
response = requests.get(f"{API_BASE_URL}/feedback", params={"limit": 10000})
response.raise_for_status()

feedback_df = pd.DataFrame(response.json())

print("Total feedback records:", len(feedback_df))

if not feedback_df.empty:
    print(feedback_df["feedback_label"].value_counts())

feedback_df.head()

Total feedback records: 125
feedback_label
not_interested    66
interested        59
Name: count, dtype: int64


,feedback_id,resume_id,job_id,feedback_label,created_at,title,company,location,apply_url
0,195,8,1361,not_interested,2026-06-29 23:52:08.797656,"Software Engineer, Agents",Mirage,"New York, NY",https://www.linkedin.com/jobs/view/software-en...
1,194,8,1436,not_interested,2026-06-29 23:52:08.784255,"AI Language Engineer, Alexa for Shopping",Amazon.com Services LLC,"Seattle, WA",https://www.indeed.com/viewjob?jk=887b2c4a2fd6...
2,193,8,1213,not_interested,2026-06-29 23:52:08.768933,Software Engineer - Remote,Optum,"Eden Prairie, MN",https://www.indeed.com/viewjob?jk=b1996d9f9404...
3,192,8,1449,not_interested,2026-06-29 23:52:08.756013,Senior Infrastructure Engineer,Bank of America,"Chandler, AZ 85224",https://www.indeed.com/viewjob?jk=37d039161848...
4,191,8,1349,not_interested,2026-06-29 23:52:08.742980,Software Engineer (Open Level),BioSpace,"New York, NY",https://www.linkedin.com/jobs/view/software-en...


## 5. Training the XGBoost Feedback-Learning Model

In this step, I train the XGBoost model using the backend endpoint:

`POST /ml/train-xgb-ranker`

The model uses feedback records stored in PostgreSQL. Each feedback record is converted into a training row by combining the resume, the job, and the matching/scoring features.

The model is trained as a supervised binary classification model:

- `interested`, `applied`, and `saved` are treated as positive labels
- `not_interested` and `rejected` are treated as negative labels

Although the model is trained as a classifier, I use its output as a ranking signal. The predicted probability becomes the **ML Preference Score**, which helps personalize job recommendations.

The training result for this run was:

| Training Detail | Value |
|---|---:|
| Model trained | True |
| Training rows | 125 |
| Positive labels | 59 |
| Negative labels | 66 |

The model was saved to:

`backend/models/xgb_job_ranker.json`

This confirms that the XGBoost model successfully trained using stored feedback records from the database.

The training output also includes feature columns and feature importance values. These help explain which resume-job matching signals influenced the model most.


In [5]:
response = requests.post(f"{API_BASE_URL}/ml/train-xgb-ranker")
response.raise_for_status()

training_result = response.json()
training_result

{'trained': True,
 'training_rows': 125,
 'positive_labels': 59,
 'negative_labels': 66,
 'model_path': 'backend\\models\\xgb_job_ranker.json',
 'feature_columns': ['ats_compatibility_score',
  'baseline_final_score',
  'semantic_score',
  'skill_score',
  'required_skill_score',
  'preferred_skill_score',
  'experience_score',
  'title_score',
  'education_score',
  'keyword_context_score',
  'location_score',
  'knockout_count',
  'matched_skills_count',
  'missing_skills_count',
  'job_skills_count',
  'missing_skill_ratio',
  'source_is_linkedin',
  'source_is_indeed',
  'title_has_data',
  'title_has_engineer',
  'title_has_analyst',
  'title_has_python',
  'title_has_software',
  'text_has_sql',
  'text_has_python',
  'text_has_etl',
  'text_has_cloud'],
 'feature_importance': [{'feature': 'baseline_final_score',
   'importance': 0.5792679786682129},
  {'feature': 'ats_compatibility_score', 'importance': 0.25372111797332764},
  {'feature': 'skill_score', 'importance': 0.058011520

## 6. Evaluating the Model with Accuracy, Precision, Recall, and F1 Score

After training the model, I evaluated its prediction performance using standard classification metrics.

The notebook reloads jobs after training so that updated ML preference scores are available. It also reloads feedback records from the backend and converts feedback labels into binary values:

| Feedback Label | Binary Class |
|---|---:|
| interested / applied / saved | 1 |
| not_interested / rejected | 0 |

The evaluation compares:

- `y_true`: actual feedback labels from the database
- `y_pred`: predicted labels based on the model score threshold

A threshold of `50` is used:

- Score greater than or equal to 50 → predicted positive
- Score less than 50 → predicted negative

The evaluation results were:

| Metric | Score | Percentage |
|---|---:|---:|
| Accuracy | 0.9840 | 98.40% |
| Precision | 0.9672 | 96.72% |
| Recall | 1.0000 | 100.00% |
| F1 Score | 0.9833 | 98.33% |

**Accuracy** measures how many total predictions were correct. The model achieved **98.40% accuracy**, meaning it correctly classified most feedback examples.

**Precision** measures how many jobs predicted as positive were actually positive. The model achieved **96.72% precision**, meaning most jobs predicted as good recommendations were truly labeled as interested.

**Recall** measures how many actual positive jobs were correctly identified. The model achieved **100.00% recall**, meaning it did not miss any positive examples in this test run.

**F1 Score** balances precision and recall. The model achieved an **F1 score of 98.33%**, showing strong performance on this feedback-based test dataset.

These results show that the XGBoost model learned patterns from the feedback labels and resume-job matching features.


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Reload jobs after training
response = requests.get(
    f"{API_BASE_URL}/jobs",
    params={
        "limit": 1000,
        "offset": 0,
        "source_filter": "all",
        "freshness": "all",
        "relevant_only": "false",
        "resume_id": RESUME_ID,
    }
)

response.raise_for_status()
jobs_after_training = response.json()

jobs_eval_df = pd.DataFrame([
    {
        "job_id": job.get("id"),
        "title": job.get("title"),
        "company": job.get("company"),
        "ats_score": job.get("ats_score"),
        "ml_preference_score": job.get("ml_preference_score"),
        "recommended_score": job.get("recommended_score"),
    }
    for job in jobs_after_training
])

# Reload feedback
response = requests.get(f"{API_BASE_URL}/feedback", params={"limit": 10000})
response.raise_for_status()
feedback_df = pd.DataFrame(response.json())

def label_to_binary(label):
    label = str(label).strip().lower().replace(" ", "_")

    if label in ["interested", "applied", "saved"]:
        return 1

    if label in ["not_interested", "rejected"]:
        return 0

    return None

feedback_df["y_true"] = feedback_df["feedback_label"].apply(label_to_binary)
feedback_df = feedback_df.dropna(subset=["y_true"]).copy()
feedback_df["y_true"] = feedback_df["y_true"].astype(int)

eval_df = feedback_df.merge(
    jobs_eval_df,
    on="job_id",
    how="inner"
)

# Prefer ML preference score after training.
if "ml_preference_score" in eval_df.columns and not eval_df["ml_preference_score"].isna().all():
    score_col = "ml_preference_score"
else:
    score_col = "recommended_score"

eval_df = eval_df.dropna(subset=[score_col]).copy()

threshold = 50
eval_df["y_pred"] = (eval_df[score_col] >= threshold).astype(int)

y_true = eval_df["y_true"]
y_pred = eval_df["y_pred"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1],
    "Percentage": [
        round(accuracy * 100, 2),
        round(precision * 100, 2),
        round(recall * 100, 2),
        round(f1 * 100, 2),
    ]
})

metrics_df

,Metric,Score,Percentage
0,Accuracy,0.984000,98.40
1,Precision,0.967213,96.72
2,Recall,1.000000,100.00
3,F1 Score,0.983333,98.33


## 7. Confusion Matrix

The confusion matrix provides a more detailed view of model performance by showing correct and incorrect predictions for each class.

The confusion matrix from this run was:

|  | Predicted Negative | Predicted Positive |
|---|---:|---:|
| Actual Negative | 64 | 2 |
| Actual Positive | 0 | 59 |

This means:

- **64 negative feedback examples** were correctly predicted as negative.
- **59 positive feedback examples** were correctly predicted as positive.
- **2 negative feedback examples** were incorrectly predicted as positive.
- **0 positive feedback examples** were incorrectly predicted as negative.

The model made two false positive predictions, meaning it predicted two jobs as positive even though they were labeled as negative.

The model made zero false negative predictions, meaning it did not miss any positive jobs in this test run.

This is useful for the project because missing strong jobs would be a bigger issue for a recommendation system. In this test, the model successfully identified all positive examples.


In [7]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

cm_df

,Predicted Negative,Predicted Positive
Actual Negative,64,2
Actual Positive,0,59


## 8. Classification Report

The classification report summarizes precision, recall, F1 score, and support for each class.

The report showed:

| Class | Precision | Recall | F1 Score | Support |
|---|---:|---:|---:|---:|
| Negative Feedback | 1.00 | 0.97 | 0.98 | 66 |
| Positive Feedback | 0.97 | 1.00 | 0.98 | 59 |

The total evaluation support was **125 feedback records**.

The results show that both classes performed well:

- The negative feedback class had perfect precision and high recall.
- The positive feedback class had high precision and perfect recall.
- Both classes achieved an F1 score of approximately **0.98**.

This confirms that the model was able to separate positive and negative job preferences in this feedback-based evaluation run.


In [8]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Negative Feedback", "Positive Feedback"],
        zero_division=0
    )
)

                   precision    recall  f1-score   support

Negative Feedback       1.00      0.97      0.98        66
Positive Feedback       0.97      1.00      0.98        59

         accuracy                           0.98       125
        macro avg       0.98      0.98      0.98       125
     weighted avg       0.98      0.98      0.98       125



## 9. Discussion and Conclusion

This notebook confirms that the OpenJobAI XGBoost feedback-learning pipeline is working end-to-end.

The completed workflow includes:

1. Loading scored jobs from the FastAPI backend.
2. Creating a bulk feedback dataset from scored jobs.
3. Saving feedback records into PostgreSQL.
4. Training the XGBoost model using backend training logic.
5. Evaluating the model using standard classification metrics.

The model successfully trained on **125 feedback records**, including **59 positive labels** and **66 negative labels**.

It achieved:

- **98.40% accuracy**
- **96.72% precision**
- **100.00% recall**
- **98.33% F1 score**

These results show that the model can learn from feedback labels and resume-job matching features.

However, the results should be interpreted carefully. The feedback dataset was created for MVP testing by labeling high-scoring jobs as positive and low-scoring jobs as negative. This proves that the ML workflow is connected and trainable, but it does not represent final production-level model performance.

A stronger evaluation would require:

- More real user feedback
- A larger labeled dataset
- A proper train-test split
- Cross-validation
- Ranking-specific metrics
- Testing on unseen jobs

Overall, this notebook demonstrates that OpenJobAI includes a functional machine learning layer. The ATS-style score provides an explainable baseline, and the XGBoost model adds a feedback-based personalization layer that can improve as more user feedback is collected.
